## 🛠️ 셀0. 세팅 & 데이터 로드

In [ ]:
# !pip install matplotlib seaborn rouge wordcloud
# !apt-get install -y fonts-nanum
# !rm -rf ~/.cache/matplotlib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib as mpl
import seaborn as sns
import re
from collections import Counter
from rouge import Rouge

In [ ]:
# 한글 폰트 설정
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
fe = fm.FontEntry(fname=font_path, name='NanumGothic')
fm.fontManager.ttflist.insert(0, fe)
plt.rcParams.update({'font.family': 'NanumGothic', 'font.size': 11})
mpl.rcParams['axes.unicode_minus'] = False

In [ ]:
# 데이터 로드
DATA_PATH = "/data/ephemeral/home/data/"
train = pd.read_csv(DATA_PATH + "train.csv")
dev   = pd.read_csv(DATA_PATH + "dev.csv")
test  = pd.read_csv(DATA_PATH + "test.csv")

print(f"Train : {len(train):,}  |  Dev : {len(dev):,}  |  Test : {len(test):,}")
print("\n[Train 컬럼]", train.columns.tolist())

for name, df in [("Train", train), ("Dev", dev), ("Test", test)]:
    print(f"{name} 결측치: {df.isnull().sum().to_dict()}")

## 📌 셀 1-1. 노이즈 패턴 정량 분석

In [ ]:
print("="*60)
print("1-1. 노이즈 패턴 정량 분석")
print("="*60)

noise_patterns = {
    "이중 역슬래시 개행 (\\\\n)": r'\\n',
    "HTML br 태그 (<br>)":         r'<br>',
    "기타 HTML 태그":               r'<[^>]+>',
    "연속 공백 (2칸 이상)":         r' {2,}',
    "특수기호(이모지 등)":          r'[^\w\s가-힣.,!?~\'"()\-:#@]',
}

results = {}
for label, pattern in noise_patterns.items():
    cnt = train['dialogue'].str.contains(pattern, regex=True, na=False).sum()
    results[label] = {'건수': cnt, '비율(%)': round(cnt/len(train)*100, 2)}

noise_df = pd.DataFrame(results).T
print(noise_df)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(noise_df.index, noise_df['비율(%)'], color='salmon', edgecolor='black')
ax.bar_label(bars, fmt='%.2f%%', padding=3)
ax.set_xlabel("전체 데이터 대비 비율 (%)")
ax.set_title("Train 대화문 노이즈 패턴별 비율")
plt.tight_layout()
plt.show()

## 📌 셀1-2. 대회/요약길이 분포

In [ ]:
print("="*60)
print("1-2. 대화 / 요약 길이 분포 (단어 수 기준)")
print("="*60)

train['dial_word_len'] = train['dialogue'].astype(str).apply(lambda x: len(x.split()))
train['summ_word_len'] = train['summary'].astype(str).apply(lambda x: len(x.split()))
dev['dial_word_len']   = dev['dialogue'].astype(str).apply(lambda x: len(x.split()))
dev['summ_word_len']   = dev['summary'].astype(str).apply(lambda x: len(x.split()))

print("\n[Train] 대화 단어 수:")
print(train['dial_word_len'].describe(percentiles=[.5,.75,.9,.95,.99]).round(1))
print("\n[Train] 요약 단어 수:")
print(train['summ_word_len'].describe(percentiles=[.5,.75,.9,.95,.99]).round(1))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, (df_name, df) in enumerate([("Train", train), ("Dev", dev)]):
    sns.histplot(df['dial_word_len'], bins=80, kde=True, ax=axes[i][0], color='steelblue')
    axes[i][0].axvline(df['dial_word_len'].quantile(0.95), color='red',
                        linestyle='--', label=f"95th={df['dial_word_len'].quantile(0.95):.0f}")
    axes[i][0].set_title(f"[{df_name}] 대화 길이 분포")
    axes[i][0].legend()

    sns.histplot(df['summ_word_len'], bins=60, kde=True, ax=axes[i][1], color='darkorange')
    axes[i][1].axvline(df['summ_word_len'].quantile(0.95), color='red',
                        linestyle='--', label=f"95th={df['summ_word_len'].quantile(0.95):.0f}")
    axes[i][1].set_title(f"[{df_name}] 요약 길이 분포")
    axes[i][1].legend()

plt.suptitle("대화 & 요약 길이 분포 (Train vs Dev)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 📌 셀1-3. 토크나이저 기준 길이 분석 ⭐️(핵심!)

In [ ]:
print("="*60)
print("1-3. 토크나이저 기준 길이 (encoder_max_len=512 검증)")
print("="*60)

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("digit82/kobart-summarization")
special_tokens = ['#Person1#','#Person2#','#Person3#','#Person4#',
                  '#Person5#','#Person6#','#Person7#',
                  '#PhoneNumber#','#Address#','#PassportNumber#']
tokenizer.add_special_tokens({'additional_special_tokens': special_tokens})

# 전체 데이터 토큰 길이 계산 (시간이 걸릴 수 있음)
sample = train.sample(min(1000, len(train)), random_state=42)
sample = sample.copy()
sample['dial_token_len'] = sample['dialogue'].apply(
    lambda x: len(tokenizer.encode(str(x), truncation=False)))
sample['summ_token_len'] = sample['summary'].apply(
    lambda x: len(tokenizer.encode(str(x), truncation=False)))

print("\n[토크나이저 기준] 대화 토큰 수:")
print(sample['dial_token_len'].describe(percentiles=[.5,.75,.9,.95,.99]).round(1))
print("\n[토크나이저 기준] 요약 토큰 수:")
print(sample['summ_token_len'].describe(percentiles=[.5,.75,.9,.95,.99]).round(1))

trunc_512 = (sample['dial_token_len'] > 512).sum()
trunc_100 = (sample['summ_token_len'] > 100).sum()
print(f"\n⚠️  대화 512토큰 초과(잘림): {trunc_512}/{len(sample)} ({trunc_512/len(sample)*100:.1f}%)")
print(f"⚠️  요약 100토큰 초과(잘림): {trunc_100}/{len(sample)} ({trunc_100/len(sample)*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(sample['dial_token_len'], bins=60, kde=True, ax=axes[0], color='steelblue')
axes[0].axvline(512, color='red', linestyle='--', linewidth=2, label='encoder_max_len=512')
axes[0].set_title("대화 토큰 수 분포 (KoBART 기준)")
axes[0].legend()

sns.histplot(sample['summ_token_len'], bins=40, kde=True, ax=axes[1], color='darkorange')
axes[1].axvline(100, color='red', linestyle='--', linewidth=2, label='decoder_max_len=100')
axes[1].set_title("요약 토큰 수 분포 (KoBART 기준)")
axes[1].legend()

plt.tight_layout()
plt.show()

## 📌 셀1-4. 화자 수 분포 & Person 토큰 빈도

In [ ]:
print("="*60)
print("1-4. 화자 수 분포 & Person 토큰 빈도")
print("="*60)

def count_speakers(text):
    speakers = set(re.findall(r'#Person\d+#', str(text)))
    return len(speakers) if speakers else 1

train['speaker_count'] = train['dialogue'].apply(count_speakers)
dev['speaker_count']   = dev['dialogue'].apply(count_speakers)

# 화자 수 분포 시각화
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for i, (name, df) in enumerate([("Train", train), ("Dev", dev)]):
    cnt = df['speaker_count'].value_counts().sort_index()
    axes[i].bar(cnt.index.astype(str), cnt.values, color='mediumseagreen', edgecolor='black')
    for j, v in zip(cnt.index, cnt.values):
        axes[i].text(str(j), v + 20, f"{v/len(df)*100:.1f}%", ha='center', fontsize=9)
    axes[i].set_title(f"[{name}] 화자 수 분포")
    axes[i].set_xlabel("화자 수")
    axes[i].set_ylabel("샘플 수")

plt.suptitle("화자 수 분포 (Train vs Dev)", fontsize=13)
plt.tight_layout()
plt.show()

# Person 토큰 등장 횟수 → Special Token 설정 근거
person_counts = {}
for i in range(1, 8):
    token = f'#Person{i}#'
    cnt = train['dialogue'].str.count(re.escape(token)).sum()
    person_counts[token] = cnt

person_series = pd.Series(person_counts)
print("\nPerson 토큰 등장 횟수:")
print(person_series)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(person_series.index, person_series.values, color='cornflowerblue', edgecolor='black')
ax.bar_label(bars, fmt='%d', padding=3)
ax.set_title("Person 토큰별 전체 등장 횟수 (Train)\n→ 등장하는 토큰만 special_tokens에 추가")
plt.tight_layout()
plt.show()

## 📌셀1-5. 턴 수 분포

In [ ]:
print("="*60)
print("1-5. 턴 수 분포")
print("="*60)

def count_turns(text):
    text = str(text).replace('\\n', '\n').replace('<br>', '\n')
    return text.count('\n') + 1

train['turn_count'] = train['dialogue'].apply(count_turns)
dev['turn_count']   = dev['dialogue'].apply(count_turns)

print("[Train] 턴 수:")
print(train['turn_count'].describe(percentiles=[.5,.75,.9,.95,.99]).round(1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, (name, df) in enumerate([("Train", train), ("Dev", dev)]):
    sns.histplot(df['turn_count'], bins=40, kde=True, ax=axes[i], color='mediumpurple')
    axes[i].axvline(df['turn_count'].median(), color='red', linestyle='--',
                    label=f"중앙값={df['turn_count'].median():.0f}")
    axes[i].set_title(f"[{name}] 턴 수 분포")
    axes[i].legend()

plt.tight_layout()
plt.show()

## 📌셀 2-1. 압축률 & 길이 상관관계

In [ ]:
print("="*60)
print("2-1. 요약 압축률 & 대화↔요약 길이 상관관계")
print("="*60)

train['compress_ratio'] = train['summ_word_len'] / train['dial_word_len']
corr = train[['dial_word_len','summ_word_len']].corr().iloc[0,1]

print(f"평균 압축률: {train['compress_ratio'].mean():.3f} "
      f"(대화의 {train['compress_ratio'].mean()*100:.1f}%만 요약)")
print(f"길이 상관계수: {corr:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(train['compress_ratio'], bins=50, kde=True, ax=axes[0], color='teal')
axes[0].axvline(train['compress_ratio'].mean(), color='red', linestyle='--',
                label=f"평균={train['compress_ratio'].mean():.2f}")
axes[0].set_title("요약 압축률 분포")
axes[0].legend()

sns.scatterplot(x=train['dial_word_len'], y=train['summ_word_len'],
                alpha=0.3, ax=axes[1], color='steelblue')
axes[1].set_title(f"대화 길이 vs 요약 길이 (상관계수={corr:.3f})")
axes[1].set_xlabel("대화 단어 수")
axes[1].set_ylabel("요약 단어 수")
plt.tight_layout()
plt.show()

## 📌 셀2-2. Word Overlap (Extractive vs Abstractive)

In [ ]:
print("="*60)
print("2-2. Word Overlap 분석 (Extractive vs Abstractive 성향)")
print("="*60)

def word_overlap(dialogue, summary):
    d = set(str(dialogue).lower().split())
    s = set(str(summary).lower().split())
    return len(d & s) / len(s) if s else 0

train['overlap'] = train.apply(lambda r: word_overlap(r['dialogue'], r['summary']), axis=1)

print(f"평균 겹침 비율: {train['overlap'].mean():.3f}")
print(f"Overlap >= 0.7 (Extractive): {(train['overlap']>=0.7).sum()/len(train)*100:.1f}%")
print(f"Overlap <  0.3 (Abstractive): {(train['overlap']< 0.3).sum()/len(train)*100:.1f}%")

fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(train['overlap'], bins=50, kde=True, color='coral', ax=ax)
ax.axvline(train['overlap'].mean(), color='red', linestyle='--',
           label=f"평균={train['overlap'].mean():.2f}")
ax.axvline(0.7, color='blue', linestyle=':', linewidth=2, label="0.7 기준선")
ax.set_title("단어 겹침 비율 분포\n(높을수록 Extractive 성향 → copy 전략 유효)")
ax.legend()
plt.tight_layout()
plt.show()

## 📌 셀2-3. 요약이 대화의 어느 위치를 반영하는가

In [ ]:
print("="*60)
print("2-3. 요약이 대화 앞/중간/끝 중 어디를 반영하는가")
print("="*60)

def position_overlap(dialogue, summary, n_parts=3):
    words = str(dialogue).split()
    part_size = max(1, len(words) // n_parts)
    s_words = set(str(summary).lower().split())
    if not s_words:
        return [0] * n_parts
    return [len(set(words[i*part_size:(i+1)*part_size]) & s_words) / len(s_words)
            for i in range(n_parts)]

pos_df = pd.DataFrame(
    train.apply(lambda r: position_overlap(r['dialogue'], r['summary']), axis=1).tolist(),
    columns=['앞부분(1/3)', '중간(2/3)', '뒷부분(3/3)']
)
print(pos_df.mean().round(3))

fig, ax = plt.subplots(figsize=(8, 5))
pos_df.mean().plot(kind='bar', ax=ax,
                   color=['steelblue','darkorange','green'], edgecolor='black')
ax.set_title("요약이 대화의 어느 위치를 반영하는가\n(높을수록 해당 위치 단어가 요약에 많이 등장)")
ax.set_ylabel("평균 단어 겹침 비율")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

## 📌2-4. 요약문 n-gram & 시작패턴

In [ ]:
print("="*60)
print("2-4. 요약문 n-gram & 시작 패턴")
print("="*60)

# 첫 어절 분포
first_words = train['summary'].astype(str).apply(lambda x: x.split()[0] if x.split() else '')
top_starts = first_words.value_counts().head(20)

# n-gram
def get_ngrams(texts, n):
    ngrams = []
    for text in texts:
        words = str(text).split()
        for i in range(len(words) - n + 1):
            ngrams.append(' '.join(words[i:i+n]))
    return Counter(ngrams).most_common(15)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 첫 어절
top_starts.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title("요약문 첫 어절 TOP 20")
axes[0].set_xlabel("첫 어절")
axes[0].tick_params(axis='x', rotation=45)

for idx, n in enumerate([2, 3]):
    ngrams = get_ngrams(train['summary'], n)
    labels = [x[0] for x in ngrams]
    counts = [x[1] for x in ngrams]
    axes[idx+1].barh(labels[::-1], counts[::-1], color='mediumseagreen', edgecolor='black')
    axes[idx+1].set_title(f"요약문 TOP 15 {n}-gram")
    axes[idx+1].set_xlabel("빈도")

plt.tight_layout()
plt.show()

## 📌 셀3. 화자 수 별 분포 & Train vs Dev 비교

In [ ]:
print("="*60)
print("3-1. 화자 수별 대화 길이 & 압축률")
print("="*60)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(x='speaker_count', y='dial_word_len',  data=train, ax=axes[0], palette='Set2')
axes[0].set_title("화자 수별 대화 길이 분포")

sns.boxplot(x='speaker_count', y='compress_ratio', data=train, ax=axes[1], palette='Set2')
axes[1].set_title("화자 수별 압축률 분포")
plt.tight_layout()
plt.show()

# ── Train vs Dev 전체 비교 ──
print("="*60)
print("3-2. Train vs Dev 분포 비교")
print("="*60)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, col, title in [
    (axes[0][0], 'dial_word_len', "대화 길이"),
    (axes[0][1], 'summ_word_len', "요약 길이"),
    (axes[1][1], 'turn_count',    "턴 수"),
]:
    ax.hist(train[col], bins=50, alpha=0.6, label='Train', color='steelblue', density=True)
    ax.hist(dev[col],   bins=50, alpha=0.6, label='Dev',   color='coral',     density=True)
    ax.set_title(f"{title} 분포 비교")
    ax.legend()

# 화자 수 비율 막대
spk_train = train['speaker_count'].value_counts(normalize=True).sort_index()
spk_dev   = dev['speaker_count'].value_counts(normalize=True).sort_index()
x = sorted(set(spk_train.index) | set(spk_dev.index))
w = 0.35
axes[1][0].bar([i-w/2 for i in range(len(x))], [spk_train.get(v,0) for v in x],
               width=w, label='Train', color='steelblue')
axes[1][0].bar([i+w/2 for i in range(len(x))], [spk_dev.get(v,0)   for v in x],
               width=w, label='Dev',   color='coral')
axes[1][0].set_xticks(range(len(x)))
axes[1][0].set_xticklabels(x)
axes[1][0].set_title("화자 수 비율 비교")
axes[1][0].legend()

plt.suptitle("Train vs Dev 분포 비교", fontsize=13)
plt.tight_layout()
plt.show()

# 중복/극단값 확인
print(f"\n중복 dialogue: {train['dialogue'].duplicated().sum()}건")
print(f"중복 summary:  {train['summary'].duplicated().sum()}건")
print(f"\n가장 긴 대화 TOP 3:")
display(train.nlargest(3, 'dial_word_len')[['fname','dial_word_len','summ_word_len']])

## 📌 셀4. 핵심 인사이트 요약

In [ ]:
print("="*60)
print("🏁 EDA 핵심 인사이트 & 다음 액션")
print("="*60)

print(f"""
📌 입력 구조
  대화 단어 수  평균={train['dial_word_len'].mean():.0f}  중앙값={train['dial_word_len'].median():.0f}
  화자 최빈값={train['speaker_count'].mode()[0]}명  턴 중앙값={train['turn_count'].median():.0f}

📌 출력 패턴
  요약 단어 수  평균={train['summ_word_len'].mean():.0f}  중앙값={train['summ_word_len'].median():.0f}
  압축률={train['compress_ratio'].mean():.3f}  Overlap={train['overlap'].mean():.3f}

📌 다음 단계 결정 사항
  ① encoder_max_len  → 토크나이저 결과 보고 재설정
  ② decoder_max_len  → 요약 95th percentile 보고 재설정
  ③ 노이즈 전처리   → \\\\n, <br> 빈도 보고 전처리 추가 여부 결정
  ④ Extractive 성향  → Overlap 높으면 copy mechanism / no_repeat_ngram 조정
  ⑤ 위치 편향       → 앞부분 반영 강하면 truncation 방향 고려
""")